In [2]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Raw data folder:", RAW_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())

# Test dataset import
list(RAW_DIR.iterdir())

Project root: C:\Projects\nid-data-mining
Raw data folder: C:\Projects\nid-data-mining\data\raw
Output folder: C:\Projects\nid-data-mining\data\outputs


[WindowsPath('../data/raw/NUSW-NB15_features.csv'),
 WindowsPath('../data/raw/UNSW-NB15_1.csv'),
 WindowsPath('../data/raw/UNSW-NB15_2.csv'),
 WindowsPath('../data/raw/UNSW-NB15_3.csv'),
 WindowsPath('../data/raw/UNSW-NB15_4.csv')]

In [4]:
# Load feature names

features_path = RAW_DIR / "NUSW-NB15_features.csv"

features = pd.read_csv(features_path, encoding="latin1")

features.columns

#Fetch feature name only

columns = features["Name"].str.strip().str.lower().tolist()

columns[:10]

['srcip',
 'sport',
 'dstip',
 'dsport',
 'proto',
 'state',
 'dur',
 'sbytes',
 'dbytes',
 'sttl']

In [5]:
# Check attack_cat distribution
files = [
    RAW_DIR / "UNSW-NB15_1.csv",
    RAW_DIR / "UNSW-NB15_2.csv",
    RAW_DIR / "UNSW-NB15_3.csv",
    RAW_DIR / "UNSW-NB15_4.csv",
]

total_counts = pd.Series(dtype="int64")

for file in files:
    print(f"Processing {file.name}...")

    for chunk in pd.read_csv(
        file,
        header=None,
        names=columns,
        chunksize=100_000,
        low_memory=False
    ):
        chunk.columns = chunk.columns.str.strip().str.lower()

        chunk["attack_cat"] = chunk["attack_cat"].fillna("Normal")
        chunk["attack_cat"] = chunk["attack_cat"].astype(str).str.strip()

        counts = chunk["attack_cat"].value_counts(dropna=False)
        total_counts = total_counts.add(counts, fill_value=0)

total_counts = total_counts.astype(int).sort_values(ascending=False)
total_counts

Processing UNSW-NB15_1.csv...
Processing UNSW-NB15_2.csv...
Processing UNSW-NB15_3.csv...
Processing UNSW-NB15_4.csv...


attack_cat
Normal            2218764
Generic            215481
Exploits            44525
Fuzzers             24246
DoS                 16353
Reconnaissance      13987
Analysis             2677
Backdoor             1795
Shellcode            1511
Backdoors             534
Worms                 174
dtype: int64

In [7]:
# Summarization of distribution
total_records = total_counts.sum()

attack_cat_distribution = pd.DataFrame({
    "Class": total_counts.index,
    "Number of Records": total_counts.values,
    "Percentage": (total_counts.values / total_records * 100).round(2)
})

attack_cat_distribution

,Class,Number of Records,Percentage
0,Normal,2218764,87.35
1,Generic,215481,8.48
2,Exploits,44525,1.75
3,Fuzzers,24246,0.95
4,DoS,16353,0.64
5,Reconnaissance,13987,0.55
6,Analysis,2677,0.11
7,Backdoor,1795,0.07
8,Shellcode,1511,0.06
9,Backdoors,534,0.02


In [8]:
# Export as csv
output_path = OUTPUT_DIR / "class_distribution_attack_cat.csv"

attack_cat_distribution.to_csv(output_path, index=False)

output_path

WindowsPath('../data/outputs/class_distribution_attack_cat.csv')